# totalVI model training on MALT CITE-seq data

This notebook loads the preprocessed MuData object generated in the first notebook and trains a totalVI model to obtain an integrated RNA + ADT protein latent representation.

## 1. Import libraries

In [1]:
import os
import tempfile
import requests

import numpy as np
import pandas as pd
import scipy

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import muon
import scvi
import torch

/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


## 2. Check package versions

In [2]:
print("scanpy:", sc.__version__)
print("scvi-tools:", scvi.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

scanpy: 1.11.5
scvi-tools: 1.3.3
torch: 2.12.0+cu130
CUDA available: True


/tmp/ipykernel_51310/203591720.py:1: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)


## 3. Load preprocessed MuData object

In [5]:
mdata = muon.read_h5mu("../data/processed/malt_citeseq_mudata.h5mu")

mdata

/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 8412 × 33555
  var:	'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'
  2 modalities
    rna:	8412 × 33538
      var:	'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'
      layers:	'counts'
    prot:	8412 × 17
      var:	'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'

In [6]:
print("RNA:", mdata.mod["rna"].shape)
print("Protein:", mdata.mod["prot"].shape)
print("RNA layers:", mdata.mod["rna"].layers.keys())

RNA: (8412, 33538)
Protein: (8412, 17)
RNA layers: KeysView(Layers with keys: counts)


## 4. Prepare AnnData object for totalVI

In [4]:
adata = mdata.mod["rna"].copy()

protein_expression = pd.DataFrame(
    mdata.mod["prot"].X.toarray() if hasattr(mdata.mod["prot"].X, "toarray") else mdata.mod["prot"].X,
    index=mdata.mod["prot"].obs_names,
    columns=mdata.mod["prot"].var_names
)

adata.obsm["protein_expression"] = protein_expression

adata

AnnData object with n_obs × n_vars = 8412 × 33538
    var: 'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'
    obsm: 'protein_expression'
    layers: 'counts'

In [7]:
print("RNA matrix:", adata.X.shape)
print("RNA counts layer:", adata.layers["counts"].shape)
print("Protein expression:", adata.obsm["protein_expression"].shape)

RNA matrix: (8412, 33538)
RNA counts layer: (8412, 33538)
Protein expression: (8412, 17)


## 5. Register AnnData with totalVI

In [10]:
adata.obs["batch"] = "MALT"

scvi.model.TOTALVI.setup_anndata(
    adata,
    layer="counts",
    batch_key="batch",
    protein_expression_obsm_key="protein_expression"
)

adata

INFO     Using column names from columns of adata.obsm['protein_expression']                                       


/tmp/ipykernel_51310/1132069409.py:3: DeprecationWarning: We recommend using setup_mudata for multi-modal data.It does not influence model performance
  scvi.model.TOTALVI.setup_anndata(


AnnData object with n_obs × n_vars = 8412 × 33538
    obs: 'batch', '_scvi_labels', '_scvi_batch'
    var: 'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'
    uns: '_scvi_uuid', '_scvi_manager_uuid'
    obsm: 'protein_expression'
    layers: 'counts'

In [13]:
adata.obs[["_scvi_batch", "_scvi_labels"]].head()

,_scvi_batch,_scvi_labels
AAACCCAAGAGGCGGA-1,0,0
AAACCCAAGCGCCTTG-1,0,0
AAACCCAAGTCTGCAT-1,0,0
AAACCCACACCATAAC-1,0,0
AAACCCAGTCTCTCCA-1,0,0


## 6. Create totalVI model

In [14]:
model = scvi.model.TOTALVI(adata)

model

INFO     Computing empirical prior initialization for protein background.                                          


TotalVI Model with the following params: 
n_latent: 20, gene_dispersion: gene, protein_dispersion: protein, gene_likelihood: nb, latent_distribution: normal
Training status: Not Trained
Model's adata is minified?: False
Model's adata is minified?: False

## 7. Train totalVI model

In [15]:
model.train(max_epochs=100)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/lightning/pytorch/core/optimizer.py:317: The lr scheduler dict contains the key(s) ['monitor'], but the keys will be ignored. You need to call `lr_scheduler.step()` manually in manual optimization.
/home/d

Training:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


In [18]:
adata.obsm["X_totalVI"] = model.get_latent_representation()

adata.obsm["X_totalVI"].shape

(8412, 20)

## 8. Save trained model and latent representation

In [19]:
adata.write_h5ad("../data/processed/malt_totalvi_trained.h5ad")
model.save("../results/models/totalvi_malt_model", overwrite=True)

In [20]:
!ls -lh ../data/processed/
!ls -lh ../results/models/

total 340M
-rw-r--r-- 1 dags dags 172M May 26 12:39 malt_citeseq_mudata.h5mu
-rw-r--r-- 1 dags dags 168M May 26 13:05 malt_totalvi_trained.h5ad
total 4.0K
drwxr-xr-x 2 dags dags 4.0K May 26 13:05 totalvi_malt_model
